In [2]:
### IMPORTAÇÃO DE BIBLIOTECAS ####

import pandas as pd
import numpy
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path
import time
import datetime as dt
import warnings

warnings.filterwarnings('ignore',category=FutureWarning)
user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"
password_encoded = quote_plus(password)
engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
    )

In [3]:
def executar_qry(query,nome_df):

    inicio = time.time()

    try:
        print(f'Executando: {nome_df}...')
        df = pd.read_sql(text(query),engine)

        tempo_total = time.time() - inicio
        minutos = int(tempo_total // 60)
        segundos = tempo_total % 60

        if minutos > 0:
            tempo = f'{minutos}min {segundos:.1f}s'
        else:
            tempo = f'{segundos:.2f}s' 

        num_linhas = len(df)

        print(f'✅ {nome_df}: {num_linhas} linhas | ⏱️{tempo}')

        return df       

    except Exception as e:
        tempo_total = time.time() - inicio
        print(f' Erro em {nome_df} após {tempo_total:.2f}s: {e}')
        return None

In [4]:
qry_sispag = """
    with Sispag AS (
SELECT
    CASE
        WHEN tipoproduto = 'C' THEN 'ARTMED360'
        ELSE 'SECAD'
    END AS "nome_ies",

    RIGHT(REPLACE(REPLACE(REPLACE(REPLACE(app_sispag_pagamento.vd_cliente.celular, '(', ''), ')', ''), '-', ''), ' ', ''), 11) AS "celular",



    vd_produto.tipoproduto::text,

    CASE
        WHEN vd_compra."codVendedor" IS NULL THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        WHEN vd_compra."codVendedor"::text = '8000'::text THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        ELSE DATE(vd_compra.datahora)
        
    END AS dt,


    TO_CHAR(DATE_TRUNC('hour', vd_compra.datahora), 'HH24:MI') AS hr,

    vd_compra.compraid AS id,
    vd_compra.clienteid AS user_id,
    
    vd_cliente.nome AS name,
    LOWER(vd_cliente.email::text) AS email,
    vd_cliente.cidade AS city,
    vd_cliente."UF" AS state,
    vd_compraitem.produtoid AS product_id,

    CASE
        WHEN vd_produto.nomeresumido::text = 'PROENDÓCRINO'::text THEN 'PROENDOCRINO'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROENF-URG'::text THEN 'PROENF/URG'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROFISIO/NEURO'::text THEN 'PROFISIO/NEF'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROFISIO/TRAUMA'::text THEN 'PROFISIO/TO'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROMEVET'::text THEN 'PROMEVET/PA'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROTERAPÊUTICA'::text THEN 'PROTERAPEUTICA'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROURGEM'::text THEN 'PROURGEN'::character varying
        WHEN vd_produto.nomeresumido::text = 'PRO-UROLOGIA'::text THEN 'PROUROLOGIA'::character varying
        ELSE vd_produto.nomeresumido
    END AS program,

    CASE
        WHEN programs.area IS NOT NULL THEN programs.area
        ELSE
            CASE
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROURGEM'::character varying::text, 'PROENDÓCRINO'::character varying::text, 'PRO-UROLOGIA'::character varying::text, 'PROTERAPÊUTICA'::character varying::text]) THEN 'Medicina'::text
                WHEN vd_produto.nomeresumido::text = 'PROMEVET'::text THEN 'Veterinária'::text
                WHEN vd_produto.nomeresumido::text = 'PROENF-URG'::text THEN 'Enfermagem'::text
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROFISIO/NEURO'::character varying::text, 'PROFISIO/TRAUMA'::character varying::text]) THEN 'Fisioterapia'::text
                ELSE NULL::text
            END
    END AS area,

    vd_compraitem.valor * vd_compra.valortotal / SUM(vd_compraitem.valor) OVER (PARTITION BY vd_compra.compraid) AS value,

    -- CORREÇÃO AQUI: Removido o 'AS' duplicado
    CASE 
        WHEN vd_compra.formapagamento = 'A' THEN 'A Vista'
        WHEN vd_compra.formapagamento = 'C' THEN 'Credito'
        WHEN vd_compra.formapagamento = 'D' THEN 'Debito'
        WHEN vd_compra.formapagamento = 'P' THEN 'Pix'
        WHEN vd_compra.formapagamento = 'R' THEN 'Recorrente'
        ELSE 'Not_found'       
    END AS payment_type,

    vd_compra."codVendedor" AS channel_id,

    CASE
        WHEN vd_compra."codVendedor" IS NULL THEN 'eCommerce'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = '8' THEN 'Representantes'
        WHEN vd_compra."codVendedor"::text IN ('9186', '9323', '9326', '9325') THEN 'Receptivo'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = 'R' THEN 'Renovacao'
        ELSE 'Call Center'
    END AS channel

FROM app_sispag_pagamento.vd_compra
LEFT JOIN app_sispag_pagamento.vd_compraitem ON vd_compra.compraid = vd_compraitem.compraid
LEFT JOIN app_sispag_pagamento.vd_produto ON vd_compraitem.produtoid = vd_produto.produtoid
LEFT JOIN app_sispag_pagamento.vd_request ON vd_compra.requestid = vd_request.requestid
LEFT JOIN app_sispag_pagamento.vd_cliente ON vd_compra.clienteid = vd_cliente.clienteid
LEFT JOIN bu_secad.programs ON vd_produto.nomeresumido::text = programs.program

WHERE DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora))) >= '2025-10-01'::date
  AND vd_produto.tipoproduto::text IN ('P', 'C')
  AND vd_request.ambiente::text = 'P'::text
  AND (vd_compra."codVendedor"::text <> '123'::text OR vd_compra."codVendedor" IS NULL)
  AND LOWER(vd_cliente.nome::text) NOT LIKE '%teste%'
)

select *    
FROM Sispag  
WHERE channel NOT IN ('eCommerce','Representantes')
and tipoproduto = 'P'
"""

qry_olos = """ 
SELECT
    call_id,
    call_type,
    customer_id,
    phone_number,
    agent_id,
    agent,
    campaign_id,
    campaign,
    tablename,
 
    -- CLASSIFICAÇÃO BASE
    CASE
        WHEN campaign = '1553' THEN 'receptivoWay'
        WHEN tablename ~* 'legado' THEN 'legado'
        WHEN tablename ~* 'evento' THEN 'evento'
        WHEN tablename ~* 'cancel' THEN 'cancelados'
        WHEN tablename ~* 'INATIVA' THEN 'inativa'
        WHEN tablename ~* 'ATIVA' THEN 'ativa'
        WHEN tablename ~* 'PAGE' THEN 'page view'
        WHEN tablename ~* 'leads' THEN 'base leads'
        ELSE NULL
    END AS base_type,
 
    -- CLASSIFICAÇÃO PÚBLICO
    CASE
        WHEN tablename ~* 'mental|psicologia' THEN 'psicologia'
        WHEN tablename ~* 'multi' THEN 'multi'
        WHEN tablename ~* 'fisio' THEN 'fisioterapia'
        WHEN tablename ~* 'enf' THEN 'enfermagem'
        WHEN tablename ~* 'medic' THEN 'medicina'
        WHEN tablename ~* 'nutri' THEN 'nutricao'
        WHEN tablename ~* 'vet' THEN 'veterinaria'
        WHEN tablename ~* 'ped' THEN 'pediatria'
        WHEN tablename ~* 'faturad' THEN 'nao faturados'
        ELSE NULL
    END AS publico,
 
    disposition_id,
    disposition_nivel_1,
 
    -- TEMPOS
    EXTRACT(EPOCH FROM agent_duration + wrap_duration) AS total_duration,
    EXTRACT(EPOCH FROM agent_duration) AS call_duration,
    EXTRACT(EPOCH FROM wrap_duration) AS tag_duration,
 
    start_agent_date::date AS data,
    EXTRACT(MONTH FROM start_agent_date) AS mes,
    start_agent_date::time AS hora,
    TO_CHAR(start_agent_date, 'HH24:00') AS hour,
 
    -- CATEGORIA MAILING
    CASE
        WHEN tablename ILIKE '%HUB%' OR campaign ILIKE '%HUB%' THEN 'REGUA'
        WHEN call_type ILIKE '%R%' THEN 'RECEPTIVO'
        WHEN call_type ILIKE '%M%' THEN 'MANUAL'
        ELSE 'MAILING'
    END AS mailing_category,
 
    -- CHAMADAS RESPONDIDAS
    CASE
        WHEN EXTRACT(EPOCH FROM wrap_duration) > 0 THEN 1
        ELSE 0
    END AS answered,
 
    ROW_NUMBER() OVER (
        PARTITION BY call_id
        ORDER BY end_agent_date
    ) AS rn
 
FROM integration_operations.vw_call_center_calls cc
WHERE start_agent_date >= '2025-09-01'
AND start_agent_date::date <= '2025-11-30'
AND campaign_id IN (1025, 1605)
ORDER BY start_agent_date DESC

"""

df = {}

try:
    queries = {
        'sispag': qry_sispag,
        'olos': qry_olos
    }
    for nome, qry in queries.items():
        df[nome] = executar_qry(qry, nome)
        
except Exception as e:
    print(f'Erro geral na execução das queries: {e}')        

Executando: sispag...
✅ sispag: 1767 linhas | ⏱️3.96s
Executando: olos...
✅ olos: 1734542 linhas | ⏱️2min 30.6s
